# PGA Schema Context
Defines the database schema and system prompts used by Claude to generate SQL.

## Imports & Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Confirm env is loaded
print("DB_HOST:", os.getenv("DB_HOST"))
print("DB_NAME:", os.getenv("DB_NAME"))
print("DB_SCHEMA:", os.getenv("DB_SCHEMA"))
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))

## Schema Context String
This is the full schema description passed to Claude on every query.

In [ ]:
DB_SCHEMA = os.getenv("DB_SCHEMA", "pga_stats")

SCHEMA_CONTEXT = f"""
You are an expert SQL analyst for a PGA Tour golf analytics database built in PostgreSQL.
You convert natural language questions into accurate PostgreSQL queries.

## DATABASE SCHEMA

### Table: {DB_SCHEMA}.combined_data
One row per golfer. Contains rolling performance metrics and composite scores.
Updated every 1-2 weeks. No tournament-level grain — player-level summary only.

Key columns:
- player (text): Golfer's full name
- composite_score (float): Weighted performance score across all metrics. Higher = better.
- composite_percentile_rank (float): Percentile rank of composite_score (0-100)
- DG_rank (int): Data Golf ranking (lower = better)
- owgr_rank (int): Official World Golf Ranking (lower = better)
- refresh_date (date): Date this row was last updated

Strokes Gained Rolling Averages (all float):
- SG_last_1 through SG_last_89: Moving average of total strokes gained over last N rounds
- SG_last_1_percentile through SG_last_89_percentile: Percentile rank of each SG_last_X column

Tournament Volume:
- total_tournaments (int): Total events played since ~2022
- tournaments_last_year (int): Events played last calendar year
- cut_percentage_last_year (float): % of events where golfer made the cut (top 50% after round 2)
- cut_percentage_last_year_percentile (float): Percentile rank of cut percentage

Finish Position Averages:
- last_3_avg_position (float): Average finish position over last 3 events
- last_5_avg_position (float): Average finish position over last 5 events
- last_10_avg_position (float): Average finish position over last 10 events

Top Finish Counts (last year):
- top_5_count, top_10_count, top_20_count (int): Count of top 5/10/20 finishes last year
- top_5_percentage_last_year, top_10_percentage_last_year, top_20_percentage_last_year (float)

Top Finish Counts (recent):
- top_5_last_5, top_10_last_5, top_20_last_5 (int): Top finishes in last 5 events
- top_5_last_10, top_10_last_10, top_20_last_10 (int): Top finishes in last 10 events
- Percentile versions: top_5_last_5_percentile, top_10_last_5_percentile, etc.

- composite_score (float): Weighted composite performance score. Higher = better.
- composite_percentile_rank (float): Percentile rank of composite_score

---

### Table: {DB_SCHEMA}.sg_data
One row per golfer per tournament. Strokes gained breakdown by event.
Only includes golfers who made the cut and completed the tournament.

Key columns:
- player (text): Golfer's full name
- tournament (text): Tournament name
- year (int): Year of the event
- week_of_season (int): Week number of the PGA season
- course (text): Course name
- avg (float): Average strokes gained per round
- total_sg_t (float): Total strokes gained overall
- total_sg_t2g (float): Strokes gained tee-to-green
- total_sg_p (float): Strokes gained putting
- signature_event (text): 'Y' if signature event, else 'N'
- major_event (text): 'Y' if major tournament, else 'N'

---

### Table: {DB_SCHEMA}.leaderboard_data
One row per golfer per tournament. Final results and scoring.

Key columns:
- player (text): Golfer's full name
- tournament (text): Tournament name
- year (int): Year of the event
- week_of_season (int): Week number of the PGA season
- course (text): Course name
- pos (text): Finishing position — can be '1', 'T3' (tied 3rd), 'CUT', 'W/D' (withdrawal)
- r1, r2, r3, r4 (float): Score to par for each round (e.g., -5 = 5 under par)
- to_par (float): Total final score to par for the tournament
- official_money (float): Prize money earned in dollars
- signature_event (text): 'Y' if signature event, else 'N'
- major_event (text): 'Y' if major tournament, else 'N'

---

## JOIN KEYS
- sg_data and leaderboard_data join on: player + tournament + year
- combined_data is player-level only; join to other tables on: player

---

## IMPORTANT SQL RULES
1. Always use schema-qualified table names: {DB_SCHEMA}.combined_data, {DB_SCHEMA}.sg_data, {DB_SCHEMA}.leaderboard_data
2. The pos column in leaderboard_data is a string. To filter actual finishes: pos NOT IN ('CUT', 'W/D')
3. To get numeric position from pos: CASE WHEN pos ~ '^T?[0-9]+$' THEN REGEXP_REPLACE(pos, '[^0-9]', '', 'g')::int ELSE NULL END
4. For money aggregations, SUM(official_money) or AVG(official_money) work directly
5. Lower finish position numbers are better (1st place = 1)
6. Higher strokes gained values are better (positive = above average)
7. Lower DG_rank and owgr_rank values are better (rank 1 = best)
8. When asked about recent form or hot players, use SG_last_5 or SG_last_10 from combined_data
9. When asked about best overall or top players, use composite_score or composite_percentile_rank
10. Always add ORDER BY and LIMIT clauses for ranking questions
11. Return only the SQL query with no explanation or markdown formatting
"""

print("SCHEMA_CONTEXT loaded successfully")
print(f"Schema name: {DB_SCHEMA}")

## System Prompts

In [ ]:
NL_TO_SQL_SYSTEM = SCHEMA_CONTEXT + """
Return ONLY the raw PostgreSQL SQL query. No markdown, no backticks, no explanation.
If the question cannot be answered with the available schema, return: ERROR: <reason>
"""

EXPLAIN_RESULTS_SYSTEM = """
You are a golf analytics expert who explains SQL query results in plain English.
Be concise, insightful, and use golf context where relevant.
Highlight the most interesting finding first.
Keep responses to 2-4 sentences unless the data warrants more detail.
Do not mention SQL or technical details — speak as an analyst presenting findings.
"""

print("System prompts ready")
print(f"NL_TO_SQL_SYSTEM length: {len(NL_TO_SQL_SYSTEM)} chars")